<a href="https://colab.research.google.com/github/munawirmt/NLP/blob/main/ENCODER_%26_DECODER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **TRANSLATION WITH SIMPLE DATASET**

In [1]:
## LIBRERIES INSTALL
!pip install tensorflow

In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
## DATASET
pairs = [
    ("I am reading", "ഞാൻ വായിക്കുന്നു"),
    ("I am coming", "ഞാൻ വരുന്നു"),
    ("Hope is life", "പ്രത്യാശയാണ് ജീവിതം"),
    ("Change addiction", "ലഹരി മാറണം"),
    ("We love coding", "ഞങ്ങൾക്ക് കോഡിംഗ് ഇഷ്ടമാണ്"),
    ("Book is knowledge", "പുസ്തകം അറിവാണ്"),
    ("Future is hope", "ഭാവി പ്രത്യാശയാണ്"),
    ("I love my college", "എനിക്ക് എന്റെ കോളേജ് ഇഷ്ടമാണ്"),
]

en_texts = [p[0] for p in pairs]
ml_texts = ["<start> " + p[1] + " <end>" for p in pairs]


In [3]:
## Tokenization
en_tokenizer = Tokenizer(filters='')
en_tokenizer.fit_on_texts(en_texts)

ml_tokenizer = Tokenizer(filters='')
ml_tokenizer.fit_on_texts(ml_texts)

en_seq = en_tokenizer.texts_to_sequences(en_texts)
ml_seq = ml_tokenizer.texts_to_sequences(ml_texts)

In [4]:
# Padding
max_en_len = max(len(s) for s in en_seq)
max_ml_len = max(len(s) for s in ml_seq)

en_seq = pad_sequences(en_seq, maxlen=max_en_len, padding='post')
ml_seq = pad_sequences(ml_seq, maxlen=max_ml_len, padding='post')

print("English vocab size:", len(en_tokenizer.word_index) + 1)
print("Malayalam vocab size:", len(ml_tokenizer.word_index) + 1)

English vocab size: 18
Malayalam vocab size: 19


In [5]:
# Decoder input and target splitTING
decoder_input_data = ml_seq[:, :-1]
decoder_target_data = ml_seq[:, 1:]

vocab_en = len(en_tokenizer.word_index) + 1
vocab_ml = len(ml_tokenizer.word_index) + 1
units = 128
embedding_dim = 64

In [6]:
## ENCODER ---
encoder_inputs = tf.keras.Input(shape=(None,))
enc_emb = tf.keras.layers.Embedding(vocab_en, embedding_dim)(encoder_inputs)
encoder_lstm = tf.keras.layers.LSTM(units, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

In [7]:
## DECODER ---
decoder_inputs = tf.keras.Input(shape=(None,))
dec_emb_layer = tf.keras.layers.Embedding(vocab_ml, embedding_dim)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = tf.keras.layers.LSTM(units, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = tf.keras.layers.Dense(vocab_ml, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [8]:
## MODEL TRAINING ---
model = tf.keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("\n--- Training Started ---")


--- Training Started ---


In [9]:
# epochs 200 TO GET GOOD TRAIN
model.fit([en_seq, decoder_input_data], np.expand_dims(decoder_target_data, -1),
          batch_size=2,
          epochs=200,
          verbose=0)

print("--- Training Done ---")

--- Training Done ---


INFERENCE MODELS

In [10]:
# Encoder model
encoder_model = tf.keras.Model(encoder_inputs, encoder_states)

In [11]:
# Decoder model
decoder_state_input_h = tf.keras.Input(shape=(units,))
decoder_state_input_c = tf.keras.Input(shape=(units,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_emb2 = dec_emb_layer(decoder_inputs)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=decoder_states_inputs)
decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)
decoder_model = tf.keras.Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2
)

TRANSLATION FUNCTION

In [12]:
def translate(english_sentence):
    # English sentence -> sequence
    seq = en_tokenizer.texts_to_sequences([english_sentence])
    seq = pad_sequences(seq, maxlen=max_en_len, padding='post')

    # TAKING context vector FROM Encoder
    states_value = encoder_model.predict(seq, verbose=0)

    # Decoder BEGIN WITH  <start>
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = ml_tokenizer.word_index['<start>']

    decoded_sentence = ''
    for _ in range(max_ml_len):
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)

        # TAKE MOST PROBABILITY GET WORD
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = ml_tokenizer.index_word.get(sampled_token_index, '')

        if sampled_word == '<end>' or sampled_word == '':
            break

        decoded_sentence += sampled_word + ' '
        # UPDATE FOR NEXT LOOP
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index
        states_value = [h, c]

    return decoded_sentence.strip()

In [13]:
# TEST
print("\n--- TEST ---")
print(f"Input: I am coming -> Output: {translate('I am coming')}")
print(f"Input: Future is hope -> Output: {translate('Future is hope')}")
print(f"Input: Change addiction -> Output: {translate('Change addiction')}")


--- TEST ---
Input: I am coming -> Output: ഞാൻ വരുന്നു
Input: Future is hope -> Output: ഭാവി പ്രത്യാശയാണ്
Input: Change addiction -> Output: ലഹരി മാറണം
